# Part 5, notebook 1 of 6: the room and the person inside it

Companion to the post *SpatialClaw for Inspection, Part 5: A Real Sandbox, and Trying to Break It*. The six notebooks follow the post in order:

| # | Notebook | Post section |
|---|---|---|
| 05a | **this one**: the container image and the cell runner | The design, The code (image and runner) |
| 05b | the host side and the loop, plus the 9 control-flow tests | The code (host side), Test the paths |
| 05c | the host trusts nothing: validating every reply | (hardening added after a security review) |
| 05d | replay of 634 recorded cells | Is the sandbox transparent? |
| 05e | 15 attacks, old kernel against container | Try to break it |
| 05f | cost, the live run, and what is not covered | What the sandbox costs, The live run, limits |

**The analogy.** The model writes Python. Running it on my own machine is like letting a lab assistant work at my desk with access to my drawers, my phone and my front-door key. The sandbox is a small room with no window and no phone. Paper goes in through a slot, answers come out through a slot. This notebook builds and inspects the room and the assistant's desk (the *runner*).

**Run from** `notebooks/spatialclaw-inspection/literate/` with the project venv. Needs Docker and the `sc-inspect-sandbox` image.

In [1]:
import subprocess, json, sys
from pathlib import Path
SANDBOX = Path("../sandbox")
r = subprocess.run(["docker", "images", "sc-inspect-sandbox", "--format", "{{.Repository}}  {{.Size}}"], capture_output=True, text=True)
print(r.stdout.strip() or "image missing: docker build -t sc-inspect-sandbox sandbox/")

sc-inspect-sandbox  390MB


## The image

Four lines that matter: a slim Python, three scientific packages (the model's code may `import numpy`), a non-root user with a fixed id, and the runner as the only entry point.

In [2]:
print((SANDBOX / "Dockerfile").read_text())

FROM python:3.12-slim
RUN pip install --no-cache-dir numpy scipy pillow
RUN useradd --no-create-home --uid 10001 --shell /usr/sbin/nologin runner
COPY runner.py /opt/runner.py
USER 10001
WORKDIR /scratch
ENTRYPOINT ["python", "-u", "/opt/runner.py"]



## The runner: a persistent notebook kernel in 90 lines

The host sends one JSON object per line, the runner answers with one JSON object per line. The namespace `NS` persists between cells, which is what makes it behave like a notebook kernel. Three details are worth reading in the source:

1. **The protocol keeps a private copy of stdout.** Before any model code runs, file descriptors 1 and 2 are pointed at `/dev/null`. The model's `print` goes into a buffer. The real stdout lives on a duplicated descriptor.
2. **The in-container alarm is a convenience, not security.** `signal.alarm(TIMEOUT)` interrupts a runaway cell, but the model's code runs in the same process and can switch the alarm off. The host's hard timeout is what actually protects.
3. **The runner's own docstring says so**: the container is the security boundary, not this file.

In [3]:
print((SANDBOX / "runner.py").read_text())

"""Cell runner that lives INSIDE the container. Written for this series.

Protocol (one JSON object per line):
  host -> runner  {"id": 7, "code": "..."}
  runner -> host  {"id": 7, "stdout": "...", "error": "...", "timed_out": false, "variables": {...}, "submission": {...}|null,
                   "images": [[title, data-url], ...]}
The namespace NS persists between cells (the 'persistent kernel'). The model's prints go to a buffer. Before any model code
runs, fd 1 and fd 2 are pointed at /dev/null and the protocol keeps a private copy of the real stdout.
NOTE: the model's code runs in THIS process, so it can still reach that private copy. The container, not this file, is the
security boundary. The host therefore treats every response as untrusted (ids, schema, size).
"""
import ast
import base64
import builtins
import contextlib
import io
import json
import os
import signal
import sys
import traceback
from pathlib import Path

_proto = os.fdopen(os.dup(1), "w", buffering=1)
_devnull 

## Talking to the runner by hand

No Python wrapper yet. Start the container with exactly the flags the post lists, write two cells to its stdin, read two answers. The second cell uses a variable from the first, which shows the namespace persists.

In [4]:
import sys
sys.path.insert(0, "../minimal_loop")
from docker_session import FLAGS, IMAGE
print("flags:", " ".join(FLAGS))

cells = [{"id": 1, "code": "x = 41\nprint('set x')"}, {"id": 2, "code": "print(x + 1)"}]
stdin = "".join(json.dumps(c) + "\n" for c in cells)
r = subprocess.run(["docker", "run", "-i", "--rm", *FLAGS, "-e", "CELL_TIMEOUT=5", IMAGE], input=stdin, capture_output=True, text=True, timeout=60)
for line in r.stdout.splitlines():
    d = json.loads(line)
    print({k: d[k] for k in ("id", "stdout", "error", "timed_out")})

flags: --network none --read-only --cap-drop ALL --security-opt no-new-privileges --pids-limit 64 --memory 1g --memory-swap 1g --cpus 1 --tmpfs /scratch:rw,size=64m,uid=10001,gid=10001,mode=0700
{'id': 1, 'stdout': 'set x\n', 'error': '', 'timed_out': False}
{'id': 2, 'stdout': '42\n', 'error': '', 'timed_out': False}


## What the room looks like from inside

The same flags, one cell that asks the container about itself: who am I, what can I do, what can I reach?

In [5]:
probe = '''
import os, socket
print("user        :", os.getuid(), "(non-root)")
print("capabilities:", [l.split()[1] for l in open("/proc/self/status") if l.startswith("CapEff")][0], "(all zero = none)")
print("docker sock :", os.path.exists("/var/run/docker.sock"))
try: open("/etc/x", "w")
except OSError as e: print("write /etc   :", e.strerror)
try: open("/scratch/ok", "w").write("x"); print("write /scratch: allowed (64 MB tmpfs)")
except OSError as e: print("write /scratch:", e.strerror)
try: socket.create_connection(("1.1.1.1", 53), 2)
except OSError as e: print("network      :", e.strerror)
'''
r = subprocess.run(["docker", "run", "-i", "--rm", *FLAGS, "-e", "CELL_TIMEOUT=10", IMAGE],
                   input=json.dumps({"id": 1, "code": probe}) + "\n", capture_output=True, text=True, timeout=60)
print(json.loads(r.stdout)["stdout"])

user        : 10001 (non-root)
capabilities: 0000000000000000 (all zero = none)
docker sock : False
write /etc   : Read-only file system
write /scratch: allowed (64 MB tmpfs)
network      : Network is unreachable



## What this notebook did not show

The wrapper that adds ids, timeouts, kills and validation is `DockerSession`, the next notebook. Here the host just trusted the answers, which is exactly what the rest of the series stops doing.